# Generation of datasets to train MACE models in parallel using a dask local cluster
This notebook shows how to take a set of structures and use pycp2k and ASE to generate a dataset for training MACE models.
The dataset contains coordinates, energies, forces, and stress tensors for all the structures in the system, plus the isolated atoms and their energies.
Basically, it does the same thing as the `generate_dataset.ipynb` notebook, but it does it in parallel using a dask local cluster.

## Import relevant modules
Apart from the usual suspects, we need to import the `mk_mace_dataset` function from the `pycp2k.workflows` module. This will allow us to generate a list of ASE Atoms objects, each with the energy, forces, and stress tensor calculated.

In [ ]:
from ase.io import write
import subprocess
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
from pycp2k.templates.FORCE_EVAL.PBE_templates import add_PBE_OT
from pycp2k.templates.PRINT.singlepoint import *
from pycp2k.workflows.mk_mace_dataset import load_dataset,get_elements,add_isolated_atoms

## Create the LOCAL Dask cluster

First, we need to define the dask cluster. In this example, we will define a local cluster with 4 workers and 2 threads per worker.

```python
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=4, threads_per_worker=2)
client = Client(cluster)
```

In [ ]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=4, threads_per_worker=2)
client = Client(cluster)
import os


# Creating the SLURM dask cluster
If you are running on a SLURM cluster, you need to create a slurm cluster instead, so that the calculations are submitted as batch jobs. The following code is tailored to the CPU nodes on the smpcluster at Lincoln. Fields that are commented out might be necessary for other clusters.

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, progress

cluster = SLURMCluster(
    queue='all',                  # Set this to your SLURM partition
    #project='your_project_code',      # Optional
    cores=8,
    memory='16GB',
    walltime='01:00:00',
    job_extra=["--exclusive"],        # Optional SLURM options
    interface='enp2s0f0',                 # Replace with your cluster's interface (e.g., ib0 or eno1)
    local_directory='$WORK/dask-workers'
)
# Scale to 10 workers
cluster.scale(jobs=5) # 5 systems in the test set

# Connect client to scheduler
client = Client(cluster)

print("Dashboard:", client.dashboard_link)

# Wrapping the cp2k calculations in a function
After creating the Dask cluster, we need to wrap the loop that runs the cp2k calculations in a function, which we will supply to Dask to run in parallel. Also, we are going to create separate directories for each cp2k calculation, so that we can investigate failed calculations. Since each calculation is in a separate directory, we are not calling cleanup() at the end of the loop.


In [ ]:
def run_cp2k(system):
    root_dir=os.getcwd()
    system_dir=os.path.join(root_dir,system.info["index"])
    os.makedirs(system_dir,exist_ok=True)
    os.chdir(system_dir)
    system.info["cp2k_dir"]=system_dir
    print(system, system.info)
    calc=CP2K(project_name=system.info["index"],run_type="ENERGY_FORCE",working_directory=system_dir,cp2k_command="cp2k.ssmp") # Need to redefine calculator every time?
    #add_xTB_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"])
    add_PBE_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"],Ignore_convergence_failure=True,max_scf=1,outer_max_scf=0)
    forces_path=add_print_singlepoint_forces(calc=calc,filename="forces",unit="EV/ANGSTROM")
    stress_path=add_print_stress_tensor(calc=calc,filename="./",unit="EV/ANGSTROM^3")
    try:
        calc.run()
        print(f"[DEBUG] calc.run() complete for {system.info['index']}")

        system.info["E"] = postprocess_energy(calc=calc)
        print(f"[DEBUG] Energy extracted for {system.info['index']}")

        forces = postprocess_forces(forces_path=forces_path)
        system.set_array("forces", forces)
        print(f"[DEBUG] Forces extracted for {system.info['index']}")

        stress = postprocess_stress(stress_path=stress_path, notation="voigt")
        system.info["stress"] = stress
        print(f"[DEBUG] Stress extracted for {system.info['index']}")

        system.info["cp2k_exit"]=0
        return system
    except Exception as e:
        print(f"Error: {e}")
        output_file=f"{calc.project_name}.out"
        subprocess.run(["tail", "-n", "30", output_file])
        system.info["cp2k_exit"]=1
        return system
    finally:
        os.chdir(root_dir)

## Loading the dataset and adding the isolated atoms
We will use the `load_dataset` function to load the dataset from the `ds.xyz` file. This function will return a list of ASE Atoms objects, each with the charge and number of electrons calculated.

```python
from pycp2k.workflows.mk_mace_dataset import load_dataset

ds=load_dataset("ds.xyz")
```

We will also use the `get_elements` function to get the set of elements in the dataset, then add them as isolated atoms.

```python
from pycp2k.workflows.mk_mace_dataset import get_elements,add_isolated_atoms

symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
```

We can now print the first structures to check that the isolated atoms have been added.

In [ ]:
ds=load_dataset("ds.xyz")
symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
for i in range(len(ds)):
    ds[i].info["index"]=f"system_{i}"

## Single point calculations and postprocessing
This is the main bit. The loop that we use in the `generate_dataset.ipynb` notebook is wrapped in a function, so here we just map the function to each element of the systems list.

In [ ]:
futures = client.map(run_cp2k, ds)
results = client.gather(futures)

In [ ]:
for result in results:
    print(result.info)
    print(result.arrays)

write("ds_cp2k.xyz", ds, format="extxyz", append=True)